In [ ]:
%load_ext manim

# Simulación de la distribución binomial mediante lanzamientos de monedas

Este proyecto utiliza **Manim** (v0.20.1) para visualizar cómo surge una distribución binomial a partir de múltiples experimentos aleatorios de lanzamientos de monedas.

Cada experimento consiste en:
- Lanzar 10 monedas.
- Contar el número de caras obtenidas (círuclos).
- Añadir ese resultado al histograma acumulado.

La animación muestra simultáneamente:
- Los lanzamientos individuales.
- La construcción empírica del histograma.
- La curva teórica de la distribución binomial.

In [ ]:
from manim import *
import numpy as np
import random
from math import comb

## Construcción dinámica del histograma

La clase `Histograma(VGroup)` genera las barras de frecuencias acumuladas.

Características:
- Cada barra representa un número posible de caras (0–10).
- Las frecuencias se actualizan tras cada experimento.
- La barra activa se resalta en dorado.

In [ ]:
# Clase para representar el histograma de frecuencias
class Histograma(VGroup):
    def __init__(self, frecuencias, base_y=-2.2, experimentos=180, escala=8.8, activo=None, **kwargs): 
        super().__init__(**kwargs) 
        total=max(sum(frecuencias),1) 

        # Crear las barras del histograma
        for k in range(11):
            # Calcula la altura de la barra proporcional a la frecuencia
            p=frecuencias[k]/total 
            
            # Ajusta la altura de la barra usando la escala y asegurando un mínimo para que sea visible
            h=max(.01, escala * frecuencias[k] / experimentos)
            
            # Color de la barra: dorado para la barra activa, azul para las demás
            color=GOLD if k==activo else BLUE_D 
            
            # Crear un rectángulo para la barra del histograma con altura proporcional a la frecuencia
            barra=Rectangle(width=.72, height=h)
            
            # Configura el color de relleno
            barra.set_fill(color, opacity=.85)
            
            # Configura el color del borde de la barra
            barra.set_stroke(WHITE, width=2)
            
            # Posiciona la barra en el lugar correcto del histograma (ejes x, y, z)
            barra.move_to([-5+k, base_y+h/2, 0])
            
            # Agrega la barra al grupo del histograma
            self.add(barra)

## Representación visual de los lanzamientos

La clase `Monedas(VGroup)` genera una representación gráfica de los 10 lanzamientos de cada experimento:

- ○ azul  → cara  
- ✕ roja  → cruz

Se organizan en dos filas de cinco monedas.

In [ ]:
# Clase para representar las monedas lanzadas
class Monedas(VGroup):
    def __init__(self,lanzamientos,**kwargs):
        super().__init__(**kwargs)
        dx,dy=0.7,0.55 # Distancia entre monedas

        # Crea las monedas
        for i,r in enumerate(lanzamientos):
            fila,col=i//5,i%5 # Calcula la fila y columna para posicionar la moneda
            
            # Marca la moneda siendo cara (r=1) o cruz (r=0)
            if r==1:
                # Dibuja un circulo azul para cara
                ficha=Circle(radius=.19, stroke_color=WHITE, stroke_width=2).set_fill(BLUE_D, opacity=1)
                # Dibuja un circulo blanco dentro de la moneda para represnetar la cara
                marca=Circle(radius=.07, stroke_color=WHITE, stroke_width=3)
            else:
                # Dibuja un circulo rojo para cruz
                ficha=Circle(radius=.19, stroke_color=WHITE, stroke_width=2).set_fill(RED_D, opacity=1)
                # Dibuja una X blanca dentro de la moneda para representar la cruz
                marca=VGroup(
                    Line(
                        UL*.08,
                        DR*.08,
                        stroke_width=4,
                        color=WHITE
                    ),
                    Line(
                        UR*.08,
                        DL*.08,
                        stroke_width=4,
                        color=WHITE
                    )
                )
            # Agrupa la ficha y la marca para formar la moneda dependiendo de si es cara o cruz
            moneda=VGroup(ficha, marca)
            
            # Coloca la moneda en su posición
            moneda.move_to(RIGHT*col*dx + DOWN*fila*dy)

            # Agrega la moneda al grupo de monedas
            self.add(moneda)

## Simulación y distribución binomial teórica

La escena `BinomialCoins(Scene)` coordina:

1. Generación de experimentos binomiales.  
2. Actualización del histograma acumulado.  
3. Movimiento del marcador hacia el número de caras observado.  
4. Superposición de la distribución binomial teórica.

La curva amarilla representa la función de probabilidad

$$
P(X=k)=\binom{10}{k}\left(\frac12\right)^{10}
$$

correspondiente a la distribución

$$
X \sim \text{Bin}(10,0.5)
$$

Esto permite comparar:

- **Distribución empírica** (histograma construido por simulación)  
- **Distribución teórica** (curva binomial)

A medida que aumentan los experimentos, ambas tienden a coincidir.

In [ ]:
%%manim -ql BinomialCoins

# Escena principal para simular el lanzamiento de monedas y mostrar el histograma de frecuencias
class BinomialCoins(Scene):
    def construct(self):
        n=10 # Número de monedas lanzadas
        experimentos=180 # Número de experimentos a simular
        base_y=-2.2 # Posición vertical base para el histograma
        escala=8.8 # Escala para ajustar la altura de las barras del histograma
        frecuencias=np.zeros(11) # Cuenta las frecuencias de cada número de caras (0 a 10)
        lanz=np.random.binomial(1,.5,n) # Simula el lanzamiento de n monedas (1 para cara, 0 para cruz)
        
        # Parte de las monedas
        monedas=Monedas(lanz).to_corner(UR,buff=.6) # Coloca las monedas en la esquina superior derecha
        self.add(monedas) # Agrega las monedas a la escena
        texto=Text("Caras:",font_size=28) # Texto para mostrar el número de caras obtenidas
        texto.next_to(monedas,DOWN,buff=.25) # Debajo de las monedas
        valor=MathTex("0").scale(.8) # Valor inicial de caras obtenidas
        valor.next_to(texto,RIGHT,buff=.2) # Lo coloca a la derecha del texto
        self.add(texto,valor) # Agrega el texto y el valor a la escena

        # Parte del histograma
        # Dibuja los ejes del histograma y los añade a la escena
        self.add(
            Line([-5,base_y,0],[5,base_y,0]),
            Line([-5,base_y,0],[-5,2.2,0])
        )
        # Dibuja las marcas de los ejes y los números correspondientes, y los añade a la escena
        marcas=VGroup()
        for k in range(11):
            tick=Line(
                [-5+k,base_y-.08,0],
                [-5+k,base_y+.08,0]
            )
            num=MathTex(str(k)).scale(.45)
            num.next_to(tick,DOWN,buff=.15)
            marcas.add(tick,num)
        self.add(marcas)
        barras=Histograma(frecuencias,escala=escala) # Crea el histograma inicial (con frecuencias en cero)
        self.add(barras) # Añade las barras a la escena
        
        # Dibuja la curva de la distribución binomial teórica y la añade a la escena
        puntos=[] 
        for k in range(11):
            p=comb(n,k)/(2**n)
            puntos.append(
                np.array([-5+k, base_y+escala*p, 0])
            )
        curva=VMobject()
        curva.set_points_smoothly(puntos)
        curva.set_stroke(YELLOW,width=4)
        self.play(Create(curva),run_time=2)

        # Dibuja el marcador para resaltar el valor actual
        marcador=Triangle().scale(.15)
        marcador.rotate(PI)
        marcador.move_to([0,1.65,0])
        self.add(marcador)
        
        # Simula los experimentos y actualiza las monedas, el valor de caras, el marcador y el histograma en cada experimento
        for _ in range(experimentos):
            lanz=np.random.binomial(1,.5,n) # Simula el lanzamiento de n monedas (1 para cara, 0 para cruz)
            caras=np.sum(lanz) # Suma el número de caras obtenidas en el lanzamiento
            frecuencias[caras]+=1 # Actualiza la frecuencia del número de caras obtenidas
            nuevas=Monedas(lanz).move_to(monedas) # Sustituye las monedas por las nuevas
            nuevo_hist=Histograma(frecuencias, escala=escala, activo=caras) # Crea un nuevo histograma con las frecuencias actualizadas y resalta la barra actual
            nuevo_valor=MathTex(str(caras)).scale(.8) # Crea un nuevo valor de caras obtenidas
            nuevo_valor.move_to(valor) # Sustituye el valor
            nuevo_marcador=marcador.copy() # Copia el marcador para moverlo a la nueva posición
            nuevo_marcador.move_to([-5+caras,1.65,0]) # Sustituye el marcador a la nueva posición
            # Anima la transformación de las monedas, el valor de caras, el marcador y el histograma a sus nuevas posiciones y valores
            self.play(
                Transform(monedas,nuevas),
                Transform(valor,nuevo_valor),
                Transform(marcador,nuevo_marcador),
                Transform(barras,nuevo_hist),
                run_time=.12
            )
        self.wait()